This notebook will be used to predict the number of frequncies of our variables of interest in a given speech.

In [108]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from pathlib import Path

In [109]:
# Load data
df = pd.read_excel("NLP_Training_Set.xlsx")

text_col = df.columns[0]
target_cols = df.columns[1:].tolist()
K = len(target_cols)

texts = df[text_col].astype(str).tolist()


Y = df[target_cols].astype(np.float32).values
Y_log = np.log1p(Y)

train_texts, val_texts, y_train, y_val = train_test_split(
    texts, Y_log, test_size=0.2, random_state=24
)

train_ds = Dataset.from_dict({"text": train_texts, "labels": y_train})
val_ds = Dataset.from_dict({"text": val_texts, "labels": y_val})

In [110]:
# Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 17/17 [00:00<00:00, 4451.44 examples/s]


In [111]:
# Model

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=K,
    problem_type="regression"
)

def data_collator(features):
    batch = {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]).float()
    }
    return batch

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [112]:
# Metrics

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.array(preds, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    pred_counts = np.expm1(preds)
    true_counts = np.expm1(labels)

    pred_counts = np.clip(pred_counts, 0, None)

    mae_overall = mean_absolute_error(true_counts, pred_counts)
    mae_per_target = np.mean(np.abs(true_counts - pred_counts), axis=0)

    metrics = {"mae_overall": mae_overall}
    for i, col in enumerate(target_cols):
        metrics[f"mae_{col}"] = float(mae_per_target[i])
    return metrics

In [113]:
# Check NaNs in text
print("NaNs in text:", df[text_col].isna().sum())

# Check NaNs in targets (count NaNs per target column)
nan_counts = df[target_cols].isna().sum().sort_values(ascending=False)
print("NaNs per target:\n", nan_counts[nan_counts > 0])

# Rows that have ANY NaN in targets
bad_rows = df[df[target_cols].isna().any(axis=1)]
print("Rows with NaN in targets:", len(bad_rows))


NaNs in text: 0
NaNs per target:
 Series([], dtype: int64)
Rows with NaN in targets: 0


In [114]:
# Train

args = TrainingArguments(
    output_dir="bert_counts_out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae_overall",
    greater_is_better=False,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

/var/folders/m3/62dc7wgd6xx5t59x6p35tnl80000gn/T/ipykernel_63020/1577020908.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/mileskeitz/Desktop/vs_code files/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Mae Overall,Mae Partisan Cue Deactivation,Mae Leaders Emoting As More Human As Opposed To Politician Like,Mae State And Local Level Focus,Mae National Pride,Mae High Dominance Politics,Mae Crisis Creibility Managerial competency
1,No log,0.527877,1.327611,0.581710,1.355143,1.447731,1.055591,1.760768,1.764720
2,No log,0.472905,1.291321,0.593249,1.267803,1.433682,1.054257,1.672395,1.726543
3,No log,0.459928,1.275322,0.593856,1.225901,1.430022,1.049979,1.623214,1.728960


/Users/mileskeitz/Desktop/vs_code files/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/mileskeitz/Desktop/vs_code files/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/mileskeitz/Desktop/vs_code files/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.4599281847476959, 'eval_mae_overall': 1.2753220796585083, 'eval_mae_Partisan_Cue_Deactivation': 0.5938563346862793, 'eval_mae_Leaders_Emoting_as_more_human_as_opposed_to_politician_like': 1.2259005308151245, 'eval_mae_State_and_local_level_focus': 1.4300224781036377, 'eval_mae_National_pride': 1.049979329109192, 'eval_mae_High_dominance_politics': 1.6232141256332397, 'eval_mae_crisis_creibility_managerial competency': 1.7289599180221558, 'eval_runtime': 0.7072, 'eval_samples_per_second': 24.038, 'eval_steps_per_second': 4.242, 'epoch': 3.0}


In [ ]:
# Inference on a new speech

device = torch.device("cpu")
model.to(device)

def read_text_file(path):
    return Path(path).read_text(encoding="utf-8")

def predict_counts_sliding(
    input_data,
    max_length=256,
    stride=128,
    round_to_int=True,
):
    path = Path(input_data)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

    speech = path.read_text(encoding="utf-8", errors="ignore")
    if not speech.strip():
        raise ValueError(f"File is empty: {path.resolve()}")

    if isinstance(input_data, (str, Path)) and Path(input_data).exists():
        speech = Path(input_data).read_text(encoding="utf-8")
    else:
        speech = str(input_data)

    enc = tokenizer(
        speech,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding="max_length",
    )

    input_ids = enc["input_ids"]          
    attention_mask = enc["attention_mask"]

    model.eval()
    chunk_preds = []

    with torch.no_grad():
        for i in range(input_ids.size(0)):
            batch = {
                "input_ids": input_ids[i:i+1].to(device),
                "attention_mask": attention_mask[i:i+1].to(device),
            }
            out = model(**batch)

            pred_log = out.logits.detach().cpu().numpy()[0]   
            pred = np.expm1(pred_log)                         
            pred = np.clip(pred, 0, None)

            chunk_preds.append(pred)

    chunk_preds = np.vstack(chunk_preds)  

    summed = chunk_preds.sum(axis=0)

    pred_int = np.rint(np.clip(summed, 0, None)).astype(int)

    return dict(zip(target_cols, pred_int))


{'Partisan_Cue_Deactivation': np.int64(3), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(19), 'State_and_local_level_focus': np.int64(10), 'National_pride': np.int64(8), 'High_dominance_politics': np.int64(21), 'crisis_creibility_managerial competency': np.int64(11)}


In [120]:
# Matt Bevin Results
mb_final = predict_counts_sliding("speeches/matt bevin/matt_bevin_final_debate.txt", max_length=256, stride=128)
print(mb_final)
mb_partisan = predict_counts_sliding("speeches/matt bevin/matt_bevin_partisan_winning_speech.txt", max_length=256, stride=128)
print(mb_partisan)
mb_tweets = predict_counts_sliding("speeches/matt bevin/matt_bevin_tweets.txt", max_length=256, stride=128)
print(mb_tweets)
mb_farms = predict_counts_sliding("speeches/matt bevin/Fancy Farms.txt", max_length=256, stride=128)
print(mb_farms)


{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(40), 'State_and_local_level_focus': np.int64(21), 'National_pride': np.int64(14), 'High_dominance_politics': np.int64(42), 'crisis_creibility_managerial competency': np.int64(22)}
{'Partisan_Cue_Deactivation': np.int64(3), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(17), 'State_and_local_level_focus': np.int64(9), 'National_pride': np.int64(6), 'High_dominance_politics': np.int64(19), 'crisis_creibility_managerial competency': np.int64(10)}
{'Partisan_Cue_Deactivation': np.int64(20), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(142), 'State_and_local_level_focus': np.int64(88), 'National_pride': np.int64(53), 'High_dominance_politics': np.int64(171), 'crisis_creibility_managerial competency': np.int64(89)}
{'Partisan_Cue_Deactivation': np.int64(1), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64

In [121]:
# Andy Beshear Results
ab_final = predict_counts_sliding("speeches/andy beshear/andy_beshear_final_debate.txt", max_length=256, stride=128)
ab_partisan = predict_counts_sliding("speeches/andy beshear/andy_beshear_partisan_winning_speech.txt", max_length=256, stride=128)
ab_tweets = predict_counts_sliding("speeches/andy beshear/andy_beshear_tweets.txt", max_length=256, stride=128)
ab_farms = predict_counts_sliding("speeches/andy beshear/Fancy Farms.txt", max_length=256, stride=128)
print(ab_final)
print(ab_partisan)
print(ab_tweets)
print(ab_farms)

{'Partisan_Cue_Deactivation': np.int64(4), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(32), 'State_and_local_level_focus': np.int64(16), 'National_pride': np.int64(11), 'High_dominance_politics': np.int64(34), 'crisis_creibility_managerial competency': np.int64(18)}
{'Partisan_Cue_Deactivation': np.int64(2), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(17), 'State_and_local_level_focus': np.int64(9), 'National_pride': np.int64(6), 'High_dominance_politics': np.int64(18), 'crisis_creibility_managerial competency': np.int64(10)}
{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(36), 'State_and_local_level_focus': np.int64(20), 'National_pride': np.int64(12), 'High_dominance_politics': np.int64(42), 'crisis_creibility_managerial competency': np.int64(23)}
{'Partisan_Cue_Deactivation': np.int64(1), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(10

In [118]:
# Martha Coakley Results
mc_macdc = predict_counts_sliding("speeches/Martha Coakley/MACDC's 2014 Convention.txt")
mc_oct_21 = predict_counts_sliding("speeches/Martha Coakley/Oct 21 Gubernatorial debate.txt")
mc_oct_23 = predict_counts_sliding("speeches/Martha Coakley/Oct 23, 2014 Massachusetts Gubernatorial Debate.txt")
mc_berkshire = predict_counts_sliding("speeches/Martha Coakley/The Berkshire Eagle.txt")
print(mc_macdc)
print(mc_oct_21)
print(mc_oct_23)
print(mc_berkshire)

{'Partisan_Cue_Deactivation': np.int64(2), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(14), 'State_and_local_level_focus': np.int64(8), 'National_pride': np.int64(5), 'High_dominance_politics': np.int64(14), 'crisis_creibility_managerial competency': np.int64(8)}
{'Partisan_Cue_Deactivation': np.int64(4), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(36), 'State_and_local_level_focus': np.int64(19), 'National_pride': np.int64(13), 'High_dominance_politics': np.int64(38), 'crisis_creibility_managerial competency': np.int64(19)}
{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(39), 'State_and_local_level_focus': np.int64(21), 'National_pride': np.int64(13), 'High_dominance_politics': np.int64(41), 'crisis_creibility_managerial competency': np.int64(21)}
{'Partisan_Cue_Deactivation': np.int64(0), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(1),

In [119]:
# Charlie Baker Results
cb_macdc = predict_counts_sliding("speeches/Charlie Baker/MACDC's 2014 Convention.txt")
cb_oct_21 = predict_counts_sliding("speeches/Charlie Baker/Oct 21 Gubernatorial debate.txt")
cb_oct_23 = predict_counts_sliding("speeches/Charlie Baker/Oct 23, 2014 Massachusetts Gubernatorial Debate.txt")
cb_berkshire = predict_counts_sliding("speeches/Charlie Baker/The Berkshire Eagle.txt")
print(cb_macdc)
print(cb_oct_21)
print(cb_oct_23)
print(cb_berkshire)

{'Partisan_Cue_Deactivation': np.int64(2), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(19), 'State_and_local_level_focus': np.int64(10), 'National_pride': np.int64(5), 'High_dominance_politics': np.int64(18), 'crisis_creibility_managerial competency': np.int64(10)}
{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(46), 'State_and_local_level_focus': np.int64(24), 'National_pride': np.int64(14), 'High_dominance_politics': np.int64(46), 'crisis_creibility_managerial competency': np.int64(25)}
{'Partisan_Cue_Deactivation': np.int64(6), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(47), 'State_and_local_level_focus': np.int64(25), 'National_pride': np.int64(15), 'High_dominance_politics': np.int64(50), 'crisis_creibility_managerial competency': np.int64(26)}
{'Partisan_Cue_Deactivation': np.int64(0), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(1

In [122]:
# Anthony Brown Results
ab_fox = predict_counts_sliding("speeches/Anthony Brown/Fox interview.txt")
print(ab_fox)
ab_guber = predict_counts_sliding("speeches/Anthony Brown/Oct 7 guber debate.txt")
print(ab_guber)

{'Partisan_Cue_Deactivation': np.int64(1), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(9), 'State_and_local_level_focus': np.int64(4), 'National_pride': np.int64(3), 'High_dominance_politics': np.int64(9), 'crisis_creibility_managerial competency': np.int64(5)}
{'Partisan_Cue_Deactivation': np.int64(7), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(50), 'State_and_local_level_focus': np.int64(25), 'National_pride': np.int64(16), 'High_dominance_politics': np.int64(49), 'crisis_creibility_managerial competency': np.int64(26)}


In [123]:
# Larry Hogan Results
lh_fox = predict_counts_sliding("speeches/Larry Hogan/Fox interview.txt")
print(lh_fox)
lh_guber = predict_counts_sliding("speeches/Larry Hogan/Oct 7 guber debate.txt")
print(lh_guber)

{'Partisan_Cue_Deactivation': np.int64(1), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(4), 'State_and_local_level_focus': np.int64(2), 'National_pride': np.int64(1), 'High_dominance_politics': np.int64(4), 'crisis_creibility_managerial competency': np.int64(2)}
{'Partisan_Cue_Deactivation': np.int64(7), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(54), 'State_and_local_level_focus': np.int64(26), 'National_pride': np.int64(16), 'High_dominance_politics': np.int64(51), 'crisis_creibility_managerial competency': np.int64(27)}


In [124]:
# Kris Kobach Results
kk_2018_debate = predict_counts_sliding("speeches/Kris Kobach/2018 Kansas Governor Debate - September 5, 2018.txt")
print(kk_2018_debate)
kk_final_debate = predict_counts_sliding("speeches/Kris Kobach/Final Kansas Governor Debate October 30, 2018.txt")
print(kk_final_debate)

{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(38), 'State_and_local_level_focus': np.int64(19), 'National_pride': np.int64(12), 'High_dominance_politics': np.int64(37), 'crisis_creibility_managerial competency': np.int64(20)}
{'Partisan_Cue_Deactivation': np.int64(4), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(32), 'State_and_local_level_focus': np.int64(16), 'National_pride': np.int64(9), 'High_dominance_politics': np.int64(32), 'crisis_creibility_managerial competency': np.int64(17)}


In [125]:
# Laura Kelly Results
lk_2018_debate = predict_counts_sliding("speeches/Laura Kelly/2018 Kansas Governor Debate - September 5, 2018.txt")
print(lk_2018_debate)
lk_final_debate = predict_counts_sliding("speeches/Laura Kelly/Final Kansas Governor Debate October 30, 2018.txt")
print(lk_final_debate)

{'Partisan_Cue_Deactivation': np.int64(3), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(24), 'State_and_local_level_focus': np.int64(12), 'National_pride': np.int64(7), 'High_dominance_politics': np.int64(23), 'crisis_creibility_managerial competency': np.int64(13)}
{'Partisan_Cue_Deactivation': np.int64(3), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(20), 'State_and_local_level_focus': np.int64(9), 'National_pride': np.int64(5), 'High_dominance_politics': np.int64(19), 'crisis_creibility_managerial competency': np.int64(10)}


In [126]:
# Sue Minter Results
sm_oct8 = predict_counts_sliding("speeches/Sue Minter/Oct 6.txt")
print(sm_oct8)

{'Partisan_Cue_Deactivation': np.int64(5), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(32), 'State_and_local_level_focus': np.int64(16), 'National_pride': np.int64(9), 'High_dominance_politics': np.int64(33), 'crisis_creibility_managerial competency': np.int64(18)}


In [127]:
# Phil Scott Results
ps_oct8 = predict_counts_sliding("speeches/Phil Scott/Oct 6.txt")
print(ps_oct8)

{'Partisan_Cue_Deactivation': np.int64(4), 'Leaders_Emoting_as_more_human_as_opposed_to_politician_like': np.int64(25), 'State_and_local_level_focus': np.int64(13), 'National_pride': np.int64(8), 'High_dominance_politics': np.int64(27), 'crisis_creibility_managerial competency': np.int64(14)}
